# NODAL Civic Vision — MobileNetV2 transfer learning

This notebook trains and evaluates the four-class prototype using the project scripts. It does not download or auto-label data. First place reviewed, licensed images in `datasets/vision/raw/<class>/...` (or create a manifest with `filepath,label,group_id`). Keep aerial FloodNet images in a separate `group_id`/source and consider evaluating them separately from street imagery. Use only verified RDD2022 D40 annotations as potholes; derive image labels carefully from TACO object annotations.

Expected classes: `pothole`, `garbage`, `waterlogging`, `damaged_infrastructure`. The test split is held out; do not use it to tune the model. With small datasets the reported metrics are exploratory, not evidence of field reliability.

In [ ]:
%pip install tensorflow==2.21.0 'Pillow>=11.0' 'numpy>=2.0' 'scikit-learn>=1.6' 'matplotlib>=3.9'
import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU devices:', tf.config.list_physical_devices('GPU'))

## Mount the project data
In Colab, mount Drive and make the project folder available at `/content/NODAL` (for example, copy/clone the reviewed project and dataset there). The raw directory should contain one subfolder per class. Do not use an unreviewed Internet image dump.

In [ ]:
from pathlib import Path
PROJECT = Path('/content/NODAL')
assert (PROJECT / 'ml/vision/prepare_dataset.py').is_file(), 'Place the NODAL project at /content/NODAL'
RAW = PROJECT / 'datasets/vision/raw'
for name in ['pothole','garbage','waterlogging','damaged_infrastructure']:
    files = [p for p in (RAW/name).rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}] if (RAW/name).exists() else []
    print(name, len(files))
    assert files, f'Missing labelled images: {name}'

## Create fixed train/validation/test splits
The split uses seed 42 and approximately 70/15/15 per class. A manifest's `group_id` keeps related images together, including across classes. Exact duplicate files are reported and deduplicated; corrupt files, missing paths, unsupported labels, and insufficient independent groups fail before the old generated split is replaced. Without `group_id`, source-level leakage may remain. Review counts before training.

In [ ]:
%cd /content/NODAL
!python ml/vision/prepare_dataset.py --input datasets/vision/raw --output datasets/vision/split

## Train and evaluate
The shared image utility decodes RGB and resizes to 224×224 using explicit Pillow bilinear interpolation, then applies Keras MobileNetV2 preprocessing. Training, validation, test, CLI, and Flask civic inference use that same path. The script freezes an ImageNet MobileNetV2 backbone, trains a small classification head, applies augmentation only to training batches, and checkpoints on validation loss. Optional fine-tuning unfreezes only the last 30 backbone layers at a lower learning rate and is kept only when validation loss improves. Evaluation uses held-out test data once. Threshold selection uses validation only: the lowest cutoff with at least 20 selected examples and 95% Wilson lower bound of top-1 accuracy ≥0.80 maximizes coverage subject to that rule. It requires at least 40 validation examples and five per class; otherwise all predictions are sent for human review.

In [ ]:
!python ml/vision/train.py --data datasets/vision/split --artifacts ml/vision/artifacts --epochs 20

In [ ]:
# Optional: only run if validation results support further tuning; test data remains untouched.
!python ml/vision/train.py --data datasets/vision/split --artifacts ml/vision/artifacts --epochs 20 --fine-tune

In [ ]:
!python ml/vision/evaluate.py --data datasets/vision/split --model ml/vision/artifacts/nodal_civic.keras --output ml/vision/artifacts
import json
report = json.loads(Path('ml/vision/artifacts/evaluation.json').read_text())
print(json.dumps(report, indent=2))

## Export to the Flask app
Copy `nodal_civic.keras`, `class_labels.json`, and `inference_config.json` into the repository's `ml/vision/artifacts/` directory. Then set `NODAL_VISION_MODE=civic` before starting Flask. The demo remains available with `NODAL_VISION_MODE=demo`. Never ship this prototype as an automated decision system; validate performance on representative, independently sourced Bengaluru images first.